<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/hvac_recall_deep_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HVAC Recall Deep Agent Demo

This notebook builds a simple Google Colab chat app powered by a LangChain Deep Agent. The agent can answer HVAC recall questions from an existing OpenAI vector store and manage a mock CSV-backed RMA workflow.

Run the cells from top to bottom, set your OpenAI API key and vector store ID, then use the chat box at the end.

In [ ]:
!pip install -qU deepagents langchain-openai langchain-core pandas ipywidgets

In [ ]:
import os
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
VECTOR_STORE_ID = userdata.get("PRODUCT_RECALL_STORE")
MODEL_NAME = userdata.get("OPENAI_MODEL_NAME")
RMA_CSV_PATH = "/content/hvac_rma_status.csv"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print(f"MODEL_NAME={MODEL_NAME}")
print(f"RMA_CSV_PATH={RMA_CSV_PATH}")

In [ ]:
import json
from datetime import datetime

import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

RMA_COLUMNS = [
    "product_id",
    "batch_number",
    "customer_name",
    "customer_contact",
    "rma_approved",
    "approval_reason",
    "replacement_status",
    "replacement_tracking",
    "approved_at",
]

SEED_RMA_ROWS = [
    {
        "product_id": "HVAC-4410",
        "batch_number": "BATCH-7781",
        "customer_name": "Northwind Mechanical",
        "customer_contact": "service@northwind.example",
        "rma_approved": True,
        "approval_reason": "Mandatory recall confirmed from product documentation.",
        "replacement_status": "Replacement shipped",
        "replacement_tracking": "1Z-7781-RECALL",
        "approved_at": "2026-04-01T09:15:00",
    },
    {
        "product_id": "HVAC-5520",
        "batch_number": "BATCH-8820",
        "customer_name": "",
        "customer_contact": "",
        "rma_approved": False,
        "approval_reason": "Pending recall review.",
        "replacement_status": "No replacement started",
        "replacement_tracking": "",
        "approved_at": "",
    },
]


def normalize_key(value: str) -> str:
    return str(value).strip()


def to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() == "true"


def seed_rma_database(force_reset: bool = False) -> pd.DataFrame:
    if force_reset or not os.path.exists(RMA_CSV_PATH):
        df = pd.DataFrame(SEED_RMA_ROWS, columns=RMA_COLUMNS)
        df.to_csv(RMA_CSV_PATH, index=False)
    return pd.read_csv(RMA_CSV_PATH).fillna("")


def load_rma_dataframe() -> pd.DataFrame:
    return pd.read_csv(RMA_CSV_PATH).fillna("")


def save_rma_dataframe(df: pd.DataFrame) -> None:
    df.to_csv(RMA_CSV_PATH, index=False)


def find_rma_index(df: pd.DataFrame, product_id: str, batch_number: str):
    product_key = normalize_key(product_id)
    batch_key = normalize_key(batch_number)
    matches = df.index[
        (df["product_id"].astype(str).str.strip() == product_key)
        & (df["batch_number"].astype(str).str.strip() == batch_key)
    ]
    return matches[0] if len(matches) else None


def extract_text_content(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, dict):
                if item.get("type") in {"text", "output_text"}:
                    text_parts.append(str(item.get("text", "")))
                elif "text" in item:
                    text_parts.append(str(item["text"]))
            else:
                text_parts.append(str(item))
        return "\n".join(part for part in text_parts if part)
    return str(content)


file_search_tool = {
    "type": "file_search",
    "vector_store_ids": [VECTOR_STORE_ID],
    "max_num_results": 5,
}

chat_model = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENAI_API_KEY,
    include=["file_search_call.results"],
)

approval_model = chat_model.bind_tools([file_search_tool])


def get_rma_status_impl(product_id: str, batch_number: str) -> dict:
    df = load_rma_dataframe()
    row_index = find_rma_index(df, product_id, batch_number)
    if row_index is None:
        return {
            "found": False,
            "product_id": normalize_key(product_id),
            "batch_number": normalize_key(batch_number),
            "approved": False,
            "reason": "No RMA record found for that product and batch.",
            "replacement_status": "unknown",
            "tracking": "",
            "record": None,
        }

    record = df.loc[row_index].to_dict()
    return {
        "found": True,
        "product_id": record["product_id"],
        "batch_number": record["batch_number"],
        "approved": to_bool(record["rma_approved"]),
        "reason": str(record["approval_reason"]),
        "replacement_status": str(record["replacement_status"]),
        "tracking": str(record["replacement_tracking"]),
        "record": record,
    }


def upsert_rma_record_impl(record: dict) -> dict:
    df = load_rma_dataframe()
    merged_record = {column: record.get(column, "") for column in RMA_COLUMNS}
    merged_record["product_id"] = normalize_key(merged_record["product_id"])
    merged_record["batch_number"] = normalize_key(merged_record["batch_number"])
    row_index = find_rma_index(df, merged_record["product_id"], merged_record["batch_number"])

    if row_index is None:
        df = pd.concat([df, pd.DataFrame([merged_record])], ignore_index=True)
    else:
        for column in RMA_COLUMNS:
            df.at[row_index, column] = merged_record[column]

    save_rma_dataframe(df)
    return {
        "saved": True,
        "product_id": merged_record["product_id"],
        "batch_number": merged_record["batch_number"],
        "record": merged_record,
    }


def evaluate_mandatory_recall(product_id: str, batch_number: str) -> dict:
    prompt = f"""
You are reviewing HVAC recall documents.
Use the hosted file search tool against the connected OpenAI vector store.
Decide whether the recall for the given product and batch is mandatory.

ProductId: {normalize_key(product_id)}
BatchNumber: {normalize_key(batch_number)}

Return JSON only with this exact schema:
{{
  "mandatory_recall": true,
  "approval_reason": "short explanation",
  "evidence": ["fact 1", "fact 2"]
}}

If the documents do not clearly show a mandatory recall for this product and batch, set mandatory_recall to false.
""".strip()

    result = approval_model.invoke(prompt)
    parsed = json.loads(extract_text_content(result.content))
    return {
        "approved": bool(parsed["mandatory_recall"]),
        "reason": parsed["approval_reason"],
        "evidence": parsed["evidence"],
    }


def approve_recall_rma_impl(
    product_id: str,
    batch_number: str,
    customer_name: str = "",
    customer_contact: str = "",
) -> dict:
    decision = evaluate_mandatory_recall(product_id, batch_number)
    if not decision["approved"]:
        return {
            "found": True,
            "approved": False,
            "reason": decision["reason"],
            "replacement_status": "No replacement started",
            "tracking": "",
            "record": None,
        }

    record = {
        "product_id": normalize_key(product_id),
        "batch_number": normalize_key(batch_number),
        "customer_name": customer_name,
        "customer_contact": customer_contact,
        "rma_approved": True,
        "approval_reason": decision["reason"],
        "replacement_status": "RMA approved - replacement pending",
        "replacement_tracking": "",
        "approved_at": datetime.now(datetime.UTC),
    }
    save_result = upsert_rma_record_impl(record)
    return {
        "found": True,
        "approved": True,
        "reason": decision["reason"],
        "replacement_status": record["replacement_status"],
        "tracking": record["replacement_tracking"],
        "record": save_result["record"],
        "evidence": decision["evidence"],
    }


@tool
def get_rma_status(product_id: str, batch_number: str) -> dict:
    """Look up a mock RMA record and its current replacement status for a recalled HVAC product."""
    return get_rma_status_impl(product_id, batch_number)


@tool
def upsert_rma_record(record: dict) -> dict:
    """Create or update a mock CSV-backed RMA record keyed by product_id and batch_number."""
    return upsert_rma_record_impl(record)


@tool
def approve_recall_rma(
    product_id: str,
    batch_number: str,
    customer_name: str = "",
    customer_contact: str = "",
) -> dict:
    """Approve an RMA only when the recall documents show a mandatory recall, then write the approved record to the CSV database."""
    return approve_recall_rma_impl(product_id, batch_number, customer_name, customer_contact)


SYSTEM_PROMPT = """
You are an HVAC product recall assistant.

Rules:
- Use the OpenAI hosted file search tool for recall facts, recall dates, affected batches, and recall policy details.
- Use the full conversation history when answering follow-up questions.
- Use get_rma_status only for current RMA and replacement data.
- Use approve_recall_rma when the user wants an approval decision or asks to approve an RMA.
- Only approve an RMA for a mandatory recall.
- Do not claim an RMA was approved unless approve_recall_rma returned approved=true.
- When recall evidence is missing or non-mandatory, say that approval cannot be granted.
- Keep answers clear and concise.
""".strip()


def build_agent():
    return create_deep_agent(
        model=chat_model,
        tools=[file_search_tool, get_rma_status, approve_recall_rma, upsert_rma_record],
        system_prompt=SYSTEM_PROMPT,
    )


conversation_history = []
thread_config = {"configurable": {"thread_id": "hvac-recall-demo"}}
seed_rma_database(force_reset=False)
agent = build_agent()


In [ ]:
def chat_turn(user_text: str) -> str:
    conversation_history.append({"role": "user", "content": user_text})
    result = agent.invoke({"messages": conversation_history}, config=thread_config)
    assistant_message = result["messages"][-1]
    assistant_text = extract_text_content(assistant_message.content)
    conversation_history.append({"role": "assistant", "content": assistant_text})
    return assistant_text


def reset_chat() -> None:
    conversation_history.clear()


def reset_demo_state() -> None:
    reset_chat()
    seed_rma_database(force_reset=True)


def render_chat_history() -> None:
    with chat_output:
        clear_output()
        for message in conversation_history:
            display(Markdown(f"**{message['role'].title()}:** {message['content']}"))


chat_input = widgets.Textarea(
    placeholder="Ask about an HVAC recall, RMA status, or approval request.",
    layout=widgets.Layout(width="100%", height="120px"),
)
send_button = widgets.Button(description="Send", button_style="primary")
reset_button = widgets.Button(description="Reset Chat")
reset_db_button = widgets.Button(description="Reset Chat + CSV")
chat_output = widgets.Output()


def on_send(_):
    user_text = chat_input.value.strip()
    if not user_text:
        return
    chat_input.value = ""
    chat_turn(user_text)
    render_chat_history()


def on_reset_chat(_):
    reset_chat()
    render_chat_history()


def on_reset_demo(_):
    reset_demo_state()
    render_chat_history()


send_button.on_click(on_send)
reset_button.on_click(on_reset_chat)
reset_db_button.on_click(on_reset_demo)

display(widgets.VBox([
    chat_input,
    widgets.HBox([send_button, reset_button, reset_db_button]),
    chat_output,
]))

render_chat_history()

## Optional quick checks

Use these helpers to inspect the seeded mock RMA database or drive a manual chat turn from code.

In [ ]:
seed_rma_database(force_reset=False)

In [ ]:
# Example manual calls:
# chat_turn("Is ProductId HVAC-4410 batch BATCH-7781 in a mandatory recall?")
# chat_turn("What is the current RMA status for that item?")
# chat_turn("Approve the RMA for HVAC-4410 batch BATCH-7781 for Northwind Mechanical.")